In [56]:
import warnings
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearnex import patch_sklearn, config_context
patch_sklearn()
warnings.filterwarnings('ignore')

Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


In [57]:
import os
ipynb_path = os.getcwd()
src_path = os.path.join(ipynb_path, 'src/')
input_path = os.path.join(ipynb_path,"input/")

In [58]:

import scipy.stats as spst

sys.path.append(src_path)

import dask
import dask.dataframe as dd
from windpowerlib.wind_speed import logarithmic_profile
from src.utils import uv_to_wsd

In [59]:
gj_y = pd.read_parquet(input_path + "train_y.parquet").rename({'end_datetime': 'dt'}, axis=1)
gj_ldaps = pd.read_parquet(input_path + "train_ldaps_gyeongju.parquet")

In [60]:
print(gj_ldaps.shape)

(235818, 15)


In [61]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.model_selection import TimeSeriesSplit

# yongmin's functions
from src.utils import DataConnector
from src.metric import NMAE
from src.data_processor import *

In [62]:
from sklearn.cluster import KMeans

# 이제 특징 생성에 클러스터링 결과는 보지 않을 예정.
def addKmeansFeature(train_data, test_data):
    pd.options.mode.chained_assignment = None

    for n_clusters in range(2, 7):  # 2부터 6까지 클러스터 생성
        kmeans = KMeans(n_clusters=n_clusters, n_init=10)

        train_data[f'cluster_{n_clusters}'] = kmeans.fit_predict(train_data[['wind_speed', 'wind_direction']])
        
        test_data[f'cluster_{n_clusters}'] = kmeans.predict(test_data[['wind_speed', 'wind_direction']])

    return train_data, test_data
from sklearn.decomposition import PCA

def addPCAFeature(train_data, test_data):
    # PCA 적용할 특징 열 선택 (u, v 성분)
    wind_features = ['storm_u_5m', 'storm_v_5m', 'wind_u_10m', 'wind_v_10m', 
                     'wind_speed', 'wind_direction']
    
    # 훈련 데이터에서 PCA 학습
    pca = PCA(n_components=2)
    pca_train = pca.fit_transform(train_data[wind_features])
    
    # 훈련 데이터에 주성분 추가
    train_data['PC1'] = pca_train[:, 0]
    train_data['PC2'] = pca_train[:, 1]
    
    # 테스트 데이터에 PCA 적용
    pca_test = pca.transform(test_data[wind_features])
    test_data['PC1'] = pca_test[:, 0]
    test_data['PC2'] = pca_test[:, 1]

    # PCA 설명력 확인
    explained_variance = pca.explained_variance_ratio_
    print(f"PC1 설명력: {explained_variance[0]}")
    print(f"PC2 설명력: {explained_variance[1]}")

    return train_data, test_data

from sklearn_extra.cluster import KMedoids

def addKMedoidsFeature(train_data, test_data):
    pd.options.mode.chained_assignment = None

    for n_clusters in range(2, 7):  # 2부터 6까지 클러스터 생성
        kmedoids = KMedoids(n_clusters=n_clusters, random_state=42)

        # 훈련 데이터에 K-Medoids 클러스터링 적용
        train_data[f'medoid_cluster_{n_clusters}'] = kmedoids.fit_predict(train_data[['wind_speed', 'wind_direction']])

        # 테스트 데이터에 학습된 K-Medoids 모델 적용
        test_data[f'medoid_cluster_{n_clusters}'] = kmedoids.predict(test_data[['wind_speed', 'wind_direction']])

    return train_data, test_data


In [63]:
import os
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

def get_trasforms_datas(merged_data, numeric_columns, target, save_path):
    # 폴더가 없으면 생성
    os.makedirs(save_path, exist_ok=True)
    
    # 스케일러 초기화
    z_scaler = StandardScaler()
    minmax_scaler = MinMaxScaler()
    
    # 학습 및 테스트 데이터 분할
    x_train = merged_data.loc[merged_data['dt'].between('2020-01-01', '2020-12-31', inclusive='left'), numeric_columns]
    x_test = merged_data.loc[merged_data['dt'].between('2021-01-01', '2022-12-31', inclusive='left'), numeric_columns]
    y_train = merged_data.loc[merged_data['dt'].between('2020-01-01', '2020-12-31', inclusive='left'), target]
    y_test = merged_data.loc[merged_data['dt'].between('2021-01-01', '2022-12-31', inclusive='left'), target]

    # Min-Max Scaling
    x_train_m = minmax_scaler.fit_transform(x_train)
    x_train_m = pd.DataFrame(x_train_m, columns=x_train.columns)
    x_test_m = minmax_scaler.transform(x_test)
    x_test_m = pd.DataFrame(x_test_m, columns=x_train.columns)
    
    # MinMax 스케일러 저장
    joblib.dump(minmax_scaler, os.path.join(save_path, 'minmax_scaler.pkl'))

    # Standard Scaling
    x_train_z = z_scaler.fit_transform(x_train)
    x_train_z = pd.DataFrame(x_train_z, columns=x_train.columns)
    x_test_z = z_scaler.transform(x_test)
    x_test_z = pd.DataFrame(x_test_z, columns=x_train.columns)
    
    # Standard 스케일러 저장
    joblib.dump(z_scaler, os.path.join(save_path, 'z_scaler.pkl'))

    return x_train, x_test, x_train_m, x_test_m, x_train_z, x_test_z, y_train, y_test

In [64]:
def save_dataframes(save_path, **dataframes):
    os.makedirs(save_path, exist_ok=True)
    for name, df in dataframes.items():
        file_path = os.path.join(save_path, f"{name}.pkl")
        df.to_pickle(file_path)
        print(f"{name} saved at {file_path}")

In [65]:
# 파이프라인을 이용하여 ldaps 데이터 변환
DataPipeline = Pipeline([
    ('uv_transform', UVTransformer('wind_u_10m', 'wind_v_10m')), 
    ('wind_transform', WindTransformer('wind_speed', 10, gj_ldaps['elevation'] + 100, gj_ldaps['surf_rough'])), 
    ('datetime', DatetimeTransformer('gj', encoding=False)),
    ('feature_engineering', FeatureTransformer()),
    ])

ldaps_transformed = DataPipeline.fit_transform(gj_ldaps)

print(ldaps_transformed.shape)


average_ldaps = ldaps_transformed.drop('turbine_id', axis=1).groupby('dt').mean()
average_ldaps.columns = average_ldaps.columns.str.replace(r'[<>\[\]]', '_', regex=True)
average_ldaps.columns = average_ldaps.columns.str.replace(r'[^\w]', '_', regex=True)
average_ldaps.columns = average_ldaps.columns.str.replace(r'__+', '_', regex=True)

average_ldaps.reset_index(inplace=True)


average_ldaps['dt'] = pd.to_datetime(average_ldaps['dt']).dt.tz_localize(None)
gj_y['dt'] = pd.to_datetime(gj_y['dt']).dt.tz_localize(None)
gj_y = gj_y.loc[gj_y['plant_name'] == '경주풍력']
avg_data = pd.merge(average_ldaps, gj_y, on='dt', how='inner')


avg_data_sorted = avg_data.sort_values(['dt', 'plant_name', 'energy_kwh'], ascending=[True, True, False])
avg_data_cleaned = avg_data_sorted.drop_duplicates(subset=['dt', 'plant_name'], keep='first')

avg_data_cleaned = avg_data.drop_duplicates(subset=['dt'], keep='first')
avg_data = avg_data_cleaned
numeric_columns = avg_data.select_dtypes(include=['number']).columns.tolist()
gj_x_train, gj_x_test, gj_x_train_m, gj_x_test_m, gj_x_train_z, gj_x_test_z, gj_y_train, gj_y_test = get_trasforms_datas(avg_data, numeric_columns, 'energy_kwh', 'src/data_gj/scaler')

gj_x_train, gj_x_test = addKmeansFeature(gj_x_train, gj_x_test)
gj_x_train_m, gj_x_test_m = addKmeansFeature(gj_x_train_m, gj_x_test_m)
gj_x_train_z, gj_x_test_z = addKmeansFeature(gj_x_train_z, gj_x_test_z)
print('kmean 적용 완료')

gj_x_train, gj_x_test = addPCAFeature(gj_x_train, gj_x_test)
gj_x_train_m, gj_x_test_m = addPCAFeature(gj_x_train_m, gj_x_test_m)
gj_x_train_z, gj_x_test_z = addPCAFeature(gj_x_train_z, gj_x_test_z)
print('pca 적용 완료')

gj_x_train, gj_x_test = addKMedoidsFeature(gj_x_train, gj_x_test)
gj_x_train_m, gj_x_test_m = addKMedoidsFeature(gj_x_train_m, gj_x_test_m)
gj_x_train_z, gj_x_test_z = addKMedoidsFeature(gj_x_train_z, gj_x_test_z)
print('kmedoid 적용 완료')


(235818, 30)
kmean 적용 완료
PC1 설명력: 0.9982665777206421
PC2 설명력: 0.000830549921374768
PC1 설명력: 0.7499995755457655
PC2 설명력: 0.11854411387391413
PC1 설명력: 0.33917607071751443
PC2 설명력: 0.32569090287750574
pca 적용 완료
kmedoid 적용 완료


In [66]:
save_dataframes(
    'src/data_gj/avg_datas',
    gj_x_train=gj_x_train,
    gj_x_test=gj_x_test,
    gj_x_train_m=gj_x_train_m,
    gj_x_test_m=gj_x_test_m,
    gj_x_train_z=gj_x_train_z,
    gj_x_test_z=gj_x_test_z,
    gj_y_train=gj_y_train,
    gj_y_test=gj_y_test
)

gj_x_train saved at src/data_gj/avg_datas/gj_x_train.pkl
gj_x_test saved at src/data_gj/avg_datas/gj_x_test.pkl
gj_x_train_m saved at src/data_gj/avg_datas/gj_x_train_m.pkl
gj_x_test_m saved at src/data_gj/avg_datas/gj_x_test_m.pkl
gj_x_train_z saved at src/data_gj/avg_datas/gj_x_train_z.pkl
gj_x_test_z saved at src/data_gj/avg_datas/gj_x_test_z.pkl
gj_y_train saved at src/data_gj/avg_datas/gj_y_train.pkl
gj_y_test saved at src/data_gj/avg_datas/gj_y_test.pkl


In [67]:
yg_y = pd.read_parquet(input_path + "train_y.parquet").rename({'end_datetime': 'dt'}, axis=1)
yg_ldaps = pd.read_parquet(input_path + "train_ldaps_yeonggwang.parquet")

In [68]:
# 파이프라인을 이용하여 ldaps 데이터 변환
DataPipeline = Pipeline([
    ('uv_transform', UVTransformer('wind_u_10m', 'wind_v_10m')), 
    ('wind_transform', WindTransformer('wind_speed', 10, yg_ldaps['elevation'] + 100, yg_ldaps['surf_rough'])), 
    ('datetime', DatetimeTransformer('yg', encoding=False)),
    ('feature_engineering', FeatureTransformer()),
    ])

ldaps_transformed = DataPipeline.fit_transform(yg_ldaps)

print(ldaps_transformed.shape)

average_ldaps = ldaps_transformed.drop('turbine_id', axis=1).groupby('dt').mean()
average_ldaps.columns = average_ldaps.columns.str.replace(r'[<>\[\]]', '_', regex=True)
average_ldaps.columns = average_ldaps.columns.str.replace(r'[^\w]', '_', regex=True)
average_ldaps.columns = average_ldaps.columns.str.replace(r'__+', '_', regex=True)

average_ldaps.reset_index(inplace=True)

average_ldaps['dt'] = pd.to_datetime(average_ldaps['dt']).dt.tz_localize(None)

yg_y['dt'] = pd.to_datetime(yg_y['dt']).dt.tz_localize(None)
yg_y = yg_y.loc[yg_y['plant_name'] == '영광풍력']

avg_data = pd.merge(average_ldaps, yg_y, on='dt', how='inner')

avg_data_sorted = avg_data.sort_values(['dt', 'plant_name', 'energy_kwh'], ascending=[True, True, False])
avg_data_cleaned = avg_data_sorted.drop_duplicates(subset=['dt', 'plant_name'], keep='first')

avg_data_cleaned = avg_data.drop_duplicates(subset=['dt'], keep='first')
avg_data = avg_data_cleaned
numeric_columns = avg_data.select_dtypes(include=['number']).columns.tolist()
yg_x_train, yg_x_test, yg_x_train_m, yg_x_test_m, yg_x_train_z, yg_x_test_z, yg_y_train, yg_y_test = get_trasforms_datas(avg_data, numeric_columns, 'energy_kwh', 'src/data_yg/scaler')

yg_x_train, yg_x_test = addKmeansFeature(yg_x_train, yg_x_test)
yg_x_train_m, yg_x_test_m = addKmeansFeature(yg_x_train_m, yg_x_test_m)
yg_x_train_z, yg_x_test_z = addKmeansFeature(yg_x_train_z, yg_x_test_z)
print('kmean 적용 완료')

yg_x_train, yg_x_test = addPCAFeature(yg_x_train, yg_x_test)
yg_x_train_m, yg_x_test_m = addPCAFeature(yg_x_train_m, yg_x_test_m)
yg_x_train_z, yg_x_test_z = addPCAFeature(yg_x_train_z, yg_x_test_z)
print('pca 적용 완료')

yg_x_train, yg_x_test = addKMedoidsFeature(yg_x_train, yg_x_test)
yg_x_train_m, yg_x_test_m = addKMedoidsFeature(yg_x_train_m, yg_x_test_m)
yg_x_train_z, yg_x_test_z = addKMedoidsFeature(yg_x_train_z, yg_x_test_z)
print('kmedoid 적용 완료')


(917070, 30)
kmean 적용 완료
PC1 설명력: 0.9969626069068909
PC2 설명력: 0.001954863080754876
PC1 설명력: 0.7320484617797179
PC2 설명력: 0.14104104431898426
PC1 설명력: 0.46370585375874857
PC2 설명력: 0.23818530022354453
pca 적용 완료
kmedoid 적용 완료


In [69]:
save_dataframes(
    'src/data_yg/avg_datas',
    yg_x_train=yg_x_train,
    yg_x_test=yg_x_test,
    yg_x_train_m=yg_x_train_m,
    yg_x_test_m=yg_x_test_m,
    yg_x_train_z=yg_x_train_z,
    yg_x_test_z=yg_x_test_z,
    yg_y_train=yg_y_train,
    yg_y_test=yg_y_test
)

yg_x_train saved at src/data_yg/avg_datas/yg_x_train.pkl
yg_x_test saved at src/data_yg/avg_datas/yg_x_test.pkl
yg_x_train_m saved at src/data_yg/avg_datas/yg_x_train_m.pkl
yg_x_test_m saved at src/data_yg/avg_datas/yg_x_test_m.pkl
yg_x_train_z saved at src/data_yg/avg_datas/yg_x_train_z.pkl
yg_x_test_z saved at src/data_yg/avg_datas/yg_x_test_z.pkl
yg_y_train saved at src/data_yg/avg_datas/yg_y_train.pkl
yg_y_test saved at src/data_yg/avg_datas/yg_y_test.pkl


In [70]:
gj_x_train_z.columns

Index(['elevation', 'land_cover', 'surf_rough', 'frictional_vmax_50m',
       'frictional_vmin_50m', 'pressure', 'relative_humid', 'specific_humid',
       'temp_air', 'storm_u_5m', 'storm_v_5m', 'wind_u_10m', 'wind_v_10m',
       'wind_speed', 'wind_direction', 'wind_speed_100m', 'wind_u_100m',
       'wind_v_100m', 'hour', 'day', 'month', 'year', 'season', 'Night',
       'density', 'shear_stress', 'wind_direction_cos', 'wind_direction_sin',
       'period_hours', 'energy_kwh', 'cluster_2', 'cluster_3', 'cluster_4',
       'cluster_5', 'cluster_6', 'PC1', 'PC2', 'medoid_cluster_2',
       'medoid_cluster_3', 'medoid_cluster_4', 'medoid_cluster_5',
       'medoid_cluster_6'],
      dtype='object')